# API参考: `Model2D`

**类路径:** `model_2d.model.Model2D`

## 目的
`Model2D` 是**二维水动力模块**的入口类。它负责管理一个`Mesh`对象，并在该网格上调用有限体积法求解器（`finite_volume_step`）来模拟二维浅水方程（2D Shallow Water Equations）。

## 继承关系
`Model2D` 继承自 `common.base_model.BaseModelComponent`，因此它可以作为一个标准组件被添加到`SimulationController`中。**（注意：当前版本的耦合框架对2D模型的支持尚不完善，例如，无法直接将1D模型的出流连接到2D模型的边界上。）**

## `__init__(self, name, mesh)`

构造函数用于创建一个`Model2D`的实例。

**参数:**
- `name` (str): (继承自`BaseModelComponent`) 组件的唯一名称。
- `mesh` (Mesh): 一个包含了完整拓扑关系的`Mesh`对象实例。

### 示例

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from model_2d.model import Model2D
from model_2d.mesh import Mesh

# 1. 创建一个简单的网格
points = [(0,0), (1,0), (1,1), (0,1)]
triangles = [(0,1,2), (0,2,3)]
mesh_obj = Mesh()
mesh_obj.build_from_points_and_triangles(points, triangles)

# 2. 创建 Model2D 实例
model_2d = Model2D(name="my_2d_domain", mesh=mesh_obj)

print(f"成功创建 Model2D: '{model_2d.name}'")

## `step(self, inflows: dict, dt: float)`

这是`Model2D`作为`BaseModelComponent`所实现的接口方法，也是其核心计算方法。它只做一件事：调用`model_2d.solver.finite_volume_step`函数来执行一步有限体积计算，更新网格中所有单元（Face）的状态（`h`, `uh`, `vh`）。

**参数:**
- `inflows` (dict): 一个字典，可以包含上游的边界条件。**（注意：当前实现主要通过`set_inflow_boundary`方法来处理边界，`inflows`字典在`step`中未被充分利用。）**
- `dt` (float): 控制器传递或用户指定的时间步长 (秒)。

### 示例

In [ ]:
# 设置初始条件 (左高右低)
for face in model_2d.mesh.faces:
    if face.centroid[0] < 0.5: face.h = 2.0
    else: face.h = 1.0

print("运行前:")
for f in model_2d.mesh.faces: print(f"  Face {f.id}, h={f.h:.3f}")

# 运行一步
model_2d.step(inflows={}, dt=0.01)

print("\n运行一步后:")
for f in model_2d.mesh.faces: print(f"  Face {f.id}, h={f.h:.3f}")

## 其他重要方法

### `set_inflow_boundary(self, inflow_q, boundary_edge_index)`
设置流量输入边界。它将指定的总流量`inflow_q`分配到指定的边界边上，作为源项添加到与之相邻的单元中。

### `get_outflow(self) -> float`
(继承自`BaseModelComponent`) 返回模型的总出流量。在当前实现中，它计算流出所有出口边界的总流量。

### 示例

In [ ]:
# 在运行step之前，可以设置一个入流边界
# 假设我们要从左侧边界(Edge 3)输入10m³/s的流量
inflow_q_val = 10.0
inflow_edge_idx = 3 # 假设我们知道这是左侧的边界边
model_2d.set_inflow_boundary(inflow_q_val, inflow_edge_idx)

print(f"在边界边 {inflow_edge_idx} 上设置了 {inflow_q_val} m³/s 的入流。")

# get_outflow的用法
total_outflow = model_2d.get_outflow()
print(f"当前计算出的总出流量为: {total_outflow:.3f} m³/s")